<a href="https://colab.research.google.com/github/ltellado/CAI5307/blob/main/project/laquer-smoke-test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part I: LAQuer Reproduction (End-to-End Pipeline Smoke Test)

## What is LAQuer?
Large Language Models (LLMs) can generate fluent answers, but verifying where their statements come from is challenging. Standard citations often point to an entire document or web page, forcing the reader to search through long text to find the supporting evidence.

**LAQuer** (introduced by Hirsch et al., ACL 2025) is an attribution method that solves this problem. Given an output claim and source documents, LAQuer asks an LLM (the **Localizer**, using GPT-4o) to extract the **exact, minimal verbatim text spans** from the sources that logically support the claim, separated by semicolons. An independent Natural Language Inference (NLI) model (the **Verifier**, using DeBERTa-v3) then checks whether the extracted snippets actually prove the claim.

---

## What is a "Smoke Test" and Why Run One?
In software and research pipelines, a **smoke test** is a minimal, end-to-end dry run executed on a single sample before launching a large-scale, costly, or time-consuming evaluation.

Instead of running all 45 test queries across multiple documents right away, this smoke test validates that:
1. **The environment is correct:** The official repository is cloned and locked to an immutable commit hash for reproducibility.
2. **Prompts match the published literature:** The instructions and 3 few-shot demonstrations align exactly with the paper.
3. **The API works deterministically:** OpenAI credentials, model parameters (GPT-4o, temperature = 0), and parsing logic execute without error.
4. **Verbatim integrity holds:** Extracted text snippets are strictly verified to exist verbatim in the source passages.
5. **The automated judge functions:** The local DeBERTa-v3 NLI model correctly classifies whether the extracted premise entails the hypothesis claim.
6. **Cost and token budgets are confirmed:** Token usage is measured to ensure the full experiment stays within budget (less than 2 tenths of a cent for this run).

---

## Pipeline Flow Summary
1. **Repository Pinning:** Checkout commit `a4affb2` to freeze code and prompt state.
2. **Prompt Audit:** Verify instruction text and 3 LFQA few-shot demonstrations.
3. **Localizer Call (GPT-4o):** Extract minimal supporting spans from candidate source passages for a sample claim.
4. **Verbatim Verification:** Programmatically check that all extracted spans are exact substrings of the source texts.
5. **NLI Verification (DeBERTa-v3):** Test whether the extracted spans logically entail the target claim.
6. **Brevity Metric:** Calculate the percentage reduction in text size (compression ratio) achieved by isolating minimal spans rather than citing full passages.

## Section 1: Environment Setup and Repository Pinning

Clone the official LAQuer repository and pins it to commit `a4affb2f39e781463a7d16cddc8baea2d162d314` (Dec 1, 2025).
- Pinning to this full SHA hash ensures our runtime uses the exact code version released by the authors.
- This cell must be re-run whenever a new Colab virtual machine is allocated.

In [13]:
# Section 1: Cross-Compatible Environment Setup & Imports
import os
import sys

# 1. Dependency check & auto-install for Colab
try:
    import openai
    import transformers
    import wikipediaapi
    import datasets
except ImportError:
    %pip install -q openai "transformers[torch]" sentencepiece wikipedia-api datasets requests

# 2. Clone repository if missing (Colab), or register existing local path
repo_paths = [
    os.path.abspath("LAQuer"),
    os.path.abspath("project/LAQuer"),
    os.path.abspath("../project/LAQuer"),
]

repo_found = False
for p in repo_paths:
    if os.path.isdir(p):
        if p not in sys.path:
            sys.path.append(p)
        repo_found = True
        break

if not repo_found:
    print("Cloning LAQuer repository...")
    !git clone https://github.com/eranhirs/LAQuer.git
    !cd LAQuer && git checkout a4affb2f39e781463a7d16cddc8baea2d162d314
    sys.path.append(os.path.abspath("LAQuer"))

print("Environment and repository paths configured successfully.")

Environment and repository paths configured successfully.


## Section 2: Inspecting and Auditing the Authors' Prompts

Load the raw prompt structures directly from the pinned repository file:
`LAQuer/src/laquer_methods/llm_method_prompts.py`

Key checks against Hirsch et al. (2025):
1. `ins_prompt`: Verifies the natural language instruction requiring verbatim extraction, minimal spans, and semicolon (`;`) delimiters.
2. `lfqa_few_shot`: Confirms the presence of exactly 3 few-shot demonstrations used for Long-Form Question Answering (LFQA).
3. Data schema: Inspects the keys (`topic`, `source_spans`, `sentence`, `spans_aligned`) to ensure compatibility with our prompt formatter.

In [14]:
import runpy

# Load the prompt definitions directly from the pinned repository
prompts = runpy.run_path("LAQuer/src/laquer_methods/llm_method_prompts.py")

print("=== INSTRUCTION PROMPT ===")
print(prompts["ins_prompt"])

print("\n=== FEW-SHOT EXAMPLES COUNT ===")
print(f"Number of LFQA few-shot examples: {len(prompts['lfqa_few_shot'])}")

print("\n=== FIRST FEW-SHOT EXAMPLE STRUCTURE ===")
ex0 = prompts["lfqa_few_shot"][0]
print("Keys in example:", list(ex0.keys()))
print("Topic:", ex0.get("topic"))
print("Sentence (Output):", ex0.get("sentence"))
print("Spans aligned (Attribution):", ex0.get("spans_aligned"))

=== INSTRUCTION PROMPT ===
You are provided with an output sentence and the source texts from which it was generated. You need to identify spans in the source from which the output sentence was generated. Copy verbatim the attributing source spans, and use a semicolon (;) as a delimiter between each consecutive span. The output sentence should be fully supported by the concatenation of the attributed source spans. IMPORTANT: Each span must be verbatim copied from the corresponding sources. Do not make any changes or paraphrases to the source spans. If necessary, you may copy multiple spans from the same or source, but avoid adding un-necessary spans and keep each span as short as possible. IMPORTANT: All details in the output sentence should be supported.

=== FEW-SHOT EXAMPLES COUNT ===
Number of LFQA few-shot examples: 3

=== FIRST FEW-SHOT EXAMPLE STRUCTURE ===
Keys in example: ['topic', 'source_spans', 'sentence', 'spans_aligned']
Topic: 743da71fe5efe24a7ffd44d87b88f826d2494d457f06

## Section 3: LAQuer Localizer Pipeline (Single-Sample Test)

This section implements the core LLM Localizer module:
1. `format_block`: Formats source documents, generated output claims, and attribution spans into the few-shot template.
2. `build_laquer_prompt`: Combines the author's instruction prompt, the 3 LFQA few-shot demonstrations, and the target sample.
3. `localize`: Sends the complete prompt to OpenAI (`gpt-4o` with `temperature=0` for deterministic outputs), splits extracted spans on semicolons (`;`), and verifies whether each extracted span is strictly verbatim.
4. Test execution: Evaluates a representative sample fact ("Reparations compensate victims for abuses suffered") against 3 source passages.

In [15]:
import os
from openai import OpenAI

# Dual-source API key lookup: Colab Secrets -> Local environment variable
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except (ImportError, ModuleNotFoundError):
    pass

if not api_key:
    api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found. Set it in Colab Secrets or as a local environment variable."
    )

client = OpenAI(api_key=api_key)

def format_block(source_texts, sentence, attribution_spans=None):
    """
    Formats a single example block following the authors' exact few-shot layout:
    Input:
    Source 1: ...
    Source 2: ...
    Output: <sentence>
    Attribution: <span 1> ; <span 2>
    """
    block = "\nInput: \n"
    for i, src in enumerate(source_texts):
        block += f"Source {i+1}: {src}\n"
    block += f"\nOutput: {sentence}\n\nAttribution: \n"
    if attribution_spans is not None:
        block += " ; ".join(attribution_spans)
    return block

def build_laquer_prompt(source_texts, claim):
    """
    Constructs the full prompt:
    1. Base instruction text
    2. Three in-context few-shot examples from the paper
    3. The query sample to be localized
    """
    prompt = prompts["ins_prompt"] + "\n\n"
    for ex in prompts["lfqa_few_shot"]:
        sources = list(ex["source_spans"].values())
        prompt += format_block(sources, ex["sentence"], ex["spans_aligned"]) + "\n"
    prompt += format_block(source_texts, claim)
    return prompt

def localize(source_texts, claim, model="gpt-4o"):
    """
    Queries GPT-4o with temperature=0 to extract minimal supporting verbatim spans.
    Parses outputs delimited by semicolons and verifies verbatim compliance against sources.
    """
    full_prompt = build_laquer_prompt(source_texts, claim)

    # Request deterministic completion
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[{"role": "user", "content": full_prompt}],
    )

    # Extract raw response and parse individual spans delimited by ';'
    raw = response.choices[0].message.content.strip()
    spans = [s.strip() for s in raw.split(";") if s.strip()]

    # Verbatim validation: verify that each returned span is an exact substring of at least one source
    verbatim_checks = [any(span in src for src in source_texts) for span in spans]

    return {
        "raw": raw,
        "spans": spans,
        "verbatim": verbatim_checks,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
    }

# --- Smoke Test Execution on a Real Sample ---
# Sample fact from the paper's LFQA evaluation set
test_claim = "Reparations compensate victims for abuses suffered."
test_sources = [
    "Reparations are broadly understood as compensation given for an abuse or injury.",
    "Unlike civil litigation or other forms of legal redress, reparations are unique because they directly address the situation of the victims.",
    "Reparations, if well designed, acknowledge victims' suffering, offer measures of redress, as well as some form of compensation for the violations suffered."
]

# Run localizer
result = localize(test_sources, test_claim)

# Display outputs and verification metrics
print("=== LOCALIZER OUTPUT ===")
print("Extracted Spans:", result["spans"])
print("Verbatim check :", result["verbatim"])
print("Tokens used    :", f"{result['prompt_tokens']} prompt, {result['completion_tokens']} completion")

=== LOCALIZER OUTPUT ===
Extracted Spans: ['Reparations are broadly understood as compensation given for an abuse or injury.']
Verbatim check : [True]
Tokens used    : 595 prompt, 15 completion


## Section 4: NLI Verifier Module (DeBERTa-v3)

This section implements the automated entailment verifier:
1. Installs Hugging Face `transformers` and loads `cross-encoder/nli-deberta-v3-base`.
2. Dynamically binds label indices from `nli_model.config.id2label` to avoid class-ordering mismatches across checkpoints.
3. Evaluates entailment between:
   - Premise: Concatenated minimal spans extracted by the GPT-4o localizer
   - Hypothesis: The target claim sentence
4. Computes attribution brevity (character-level compression ratio vs. full source text).

In [16]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load the DeBERTa-v3 NLI cross-encoder checkpoint and tokenizer
NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
nli_model.eval()

def verify_entailment(premise, hypothesis):
    """
    Evaluates whether the premise entails the hypothesis using DeBERTa-v3.
    Dynamically maps class indices from model.config.id2label.
    Returns:
        predicted_label: 'entailment', 'neutral', or 'contradiction'
        is_entailed: True if top-1 predicted class is 'entailment'
        entailment_prob: float probability of entailment (0.0 to 1.0)
        probs: dictionary of normalized probabilities for each class
    """
    inputs = tokenizer(
        premise,
        hypothesis,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = nli_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]

    pred_idx = torch.argmax(probs).item()
    predicted_label = nli_model.config.id2label[pred_idx].lower()

    prob_dict = {
        nli_model.config.id2label[i].lower(): round(probs[i].item(), 4)
        for i in range(len(probs))
    }

    return {
        "predicted_label": predicted_label,
        "is_entailed": predicted_label == "entailment",
        "entailment_prob": prob_dict.get("entailment", 0.0),
        "probs": prob_dict
    }

# --- Run Verification on Localizer Output ---
premise = " ".join(result["spans"])
hypothesis = test_claim
verification = verify_entailment(premise, hypothesis)

# Calculate character-level compression ratio
total_source_chars = sum(len(src) for src in test_sources)
extracted_chars = sum(len(span) for span in result["spans"])
compression_ratio = round((1 - (extracted_chars / total_source_chars)) * 100, 2)

print("=== SMOKE TEST AUDIT ===")
print("Premise (Extracted) :", premise)
print("Hypothesis (Claim)  :", hypothesis)
print("Predicted Label     :", verification["predicted_label"].upper())
print("Entailment Conf     :", f"{verification['entailment_prob'] * 100}%")
print("All Class Probs     :", verification["probs"])
print("---")
print(f"Full source text    : {total_source_chars} characters")
print(f"Extracted span text : {extracted_chars} characters")
print(f"Attribution brevity : {compression_ratio}% reduction in text size")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

=== SMOKE TEST AUDIT ===
Premise (Extracted) : Reparations are broadly understood as compensation given for an abuse or injury.
Hypothesis (Claim)  : Reparations compensate victims for abuses suffered.
Predicted Label     : ENTAILMENT
Entailment Conf     : 98.88%
All Class Probs     : {'contradiction': 0.0001, 'entailment': 0.9888, 'neutral': 0.0111}
---
Full source text    : 374 characters
Extracted span text : 80 characters
Attribution brevity : 78.61% reduction in text size


# Part II: LAQuer Extension (Post-Hoc Wikipedia Pipeline Test)

In content-grounded QA (the original LAQuer setting), the model is given reference documents upfront. These sections test our proposed **extension**: auditing an answer generated **without reference documents (closed-book)** by searching live Wikipedia after the fact (*post-hoc*).

## What is ASQA and "Gold Wikipedia Pages"?
We evaluate on **ASQA** (Ambiguous Question Answering; Stelmakh et al., 2022), a benchmark of inherently ambiguous web queries where human annotators wrote multi-faceted answers.
* **ASQA Gold Wikipedia Pages (`wikipages`)**: The human-annotated ground-truth Wikipedia articles that benchmark authors used to verify their answers. We treat these gold titles as our retrieval target to evaluate whether automated post-hoc retrieval found the right articles or suffered a retrieval miss.

## Section 5: Environment & Wikipedia Retrieval Helpers

This section sets up the retrieval components for the extension:
1. Installs `wikipedia-api` and `datasets` for querying live Wikipedia and loading benchmark questions.
2. Implements `wiki_search`: Calls the public MediaWiki REST API to find top-ranking article titles for an atomic claim.
3. Implements `wiki_paragraphs`: Extracts clean paragraphs from a retrieved article.
4. Implements `top_passages`: Selects the top-3 candidate passages using lexical overlap as an initial coarse-filtering stage.

In [17]:
import requests
import re
import wikipediaapi

# User-Agent string required by Wikimedia API policy
USER_AGENT = "LAQuerCourseProject/0.1 (tellado@usf.edu)"

# Initialize Wikipedia API client
wiki = wikipediaapi.Wikipedia(user_agent=USER_AGENT, language="en")

def wiki_search(query, top_k=2):
    """
    Queries the MediaWiki search API to retrieve the most relevant Wikipedia article titles.
    """
    url = "https://en.wikipedia.org/w/api.php"
    headers = {"User-Agent": USER_AGENT}
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "srlimit": top_k,
        "format": "json"
    }
    response = requests.get(url, headers=headers, params=params).json()
    hits = response.get("query", {}).get("search", [])
    return [hit["title"] for hit in hits]

def wiki_paragraphs(title, min_char_length=80):
    """
    Fetches the raw text of a Wikipedia page and splits it into substantial paragraphs.
    """
    page = wiki.page(title)
    if not page.exists():
        return []
    paragraphs = [p.strip() for p in page.text.split("\n") if len(p.strip()) >= min_char_length]
    return paragraphs

def top_passages(paragraphs, claim, top_k=3):
    """
    Selects top candidate passages based on lexical word overlap with the claim.
    Acts as coarse passage filtering prior to fine-grained LAQuer span localization.
    """
    claim_words = set(re.findall(r"\w+", claim.lower()))
    scored = []
    for p in paragraphs:
        p_words = set(re.findall(r"\w+", p.lower()))
        overlap = len(claim_words & p_words)
        scored.append((overlap, p))

    # Sort descending by word overlap
    scored.sort(key=lambda x: -x[0])
    return [p for score, p in scored[:top_k]]

print("Wikipedia helpers initialized successfully.")

Wikipedia helpers initialized successfully.


## Section 6: Extension Pipeline Walkthrough (5 Stages + Audit)

### 1. **Stage 1 (Closed-Book Generation):**
   * We prompt `gpt-4o-mini` with an ambiguous question with **zero reference texts**, forcing it to generate an answer purely from its internal parametric weights.
### 2. **Stage 2 (Atomic Claim Decomposition):**
   * Multi-sentence answers bundle several independent facts. We prompt `gpt-4o-mini` to decompose the answer into standalone atomic claims and resolve pronouns (e.g., replacing "he" or "they" with explicit entity names) so each claim can be audited independently.
### 3. **Stage 3 (Post-Hoc Wikipedia Retrieval):**
   * Unlike content-grounded RAG where passages are pre-supplied, we send the selected atomic claim to the live MediaWiki Search API.
   * We fetch candidate articles, extract body paragraphs, and rank the top candidate passages by lexical overlap.
### 4. **Stage 4 (LAQuer Span Localization):**
   * We feed the retrieved Wikipedia passages into LAQuer's few-shot GPT-4o Localizer to extract the minimal verbatim supporting span (sentence or clause) that substantiates the claim.
### 5. **Stage 5 (DeBERTa-v3 NLI Verification):**
   * We evaluate natural language inference using `cross-encoder/nli-deberta-v3-base`.
   * The extracted Wikipedia span serves as the **Premise** and the generated claim serves as the **Hypothesis** to determine whether the text proves (entails) the claim.
### 6. **Error Analysis Audit:**
   * We compare the retrieved Wikipedia articles against the **ASQA Gold Wikipedia Pages** to diagnose the failure mode:
     * **Fully Supported**: Evidence was retrieved and DeBERTa confirms entailment.
     * **Retrieval Miss**: The search query retrieved biography or entity pages instead of broad comparative record lists.
     * **Temporal Drift / Scope Divergence**: Live Wikipedia facts have updated since the model's pretraining cutoff.
     * **Parametric Hallucination**: Model generated an unsupported or contradicted assertion.

In [18]:
import json
from datasets import load_dataset

# 1. Load an evaluation sample from ASQA dev split
# Using din0s/asqa (the standard ASQA dataset on Hugging Face Hub)
try:
    asqa_dataset = load_dataset("din0s/asqa", split="dev")
    sample = asqa_dataset[0]
    question = sample["ambiguous_question"]
    gold_wikipages = sample.get("wikipages", [])
except Exception as e:
    # Graceful fallback to a documented ASQA dev question if offline/rate-limited
    print(f"Notice: Loading fallback ASQA sample due to: {e}")
    question = "When did the movie The Parent Trap come out?"
    gold_wikipages = ["The Parent Trap (1961 film)", "The Parent Trap (1998 film)"]

print("=== STAGE 1: CLOSED-BOOK GENERATION ===")
print("Question:", question)

gen_response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": f"Answer the following question in a concise, factual paragraph using only your internal knowledge. Do not use outside search.\n\nQuestion: {question}"
        }
    ],
)
answer = gen_response.choices[0].message.content.strip()
print("Generated Answer:\n", answer)

print("\n=== STAGE 2: ATOMIC CLAIM DECOMPOSITION ===")
decomp_prompt = (
    "Decompose the following answer into standalone, atomic factual claims. "
    "Resolve all pronouns (e.g., 'it', 'they', 'he') into explicit entity names, dates, or titles "
    "so that each claim can be verified independently out of context. "
    "Return only a valid JSON list of strings, with no surrounding markdown or explanation.\n\n"
    f"Question: {question}\nAnswer: {answer}"
)

decomp_response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[{"role": "user", "content": decomp_prompt}],
)

raw_claims = decomp_response.choices[0].message.content.strip()
# Clean any markdown code fence formatting if present
clean_claims = re.sub(r"^```(json)?|```$", "", raw_claims).strip()
claims = json.loads(clean_claims)

print(f"Decomposed into {len(claims)} atomic claims:")
for i, c in enumerate(claims, 1):
    print(f"  Claim {i}: {c}")

# Select the first atomic claim for targeted post-hoc audit
target_claim = claims[0]
print(f"\nTarget Claim for Audit: \"{target_claim}\"")

print("\n=== STAGE 3: POST-HOC WIKIPEDIA RETRIEVAL ===")
retrieved_titles = wiki_search(target_claim, top_k=2)
print("Retrieved Wikipedia Titles:", retrieved_titles)
print("ASQA Gold Wikipedia Pages :", gold_wikipages)

all_paragraphs = []
for title in retrieved_titles:
    all_paragraphs.extend(wiki_paragraphs(title))

candidate_sources = top_passages(all_paragraphs, target_claim, top_k=3)
print(f"Extracted {len(candidate_sources)} candidate passages ({sum(len(p) for p in candidate_sources)} total chars).")

print("\n=== STAGE 4: LAQUER SPAN LOCALIZATION ===")
if candidate_sources:
    loc_result = localize(candidate_sources, target_claim)
    extracted_spans = loc_result["spans"]
    verbatim_status = loc_result["verbatim"]
    premise = " ".join(extracted_spans)
else:
    extracted_spans = []
    verbatim_status = []
    premise = ""

print("Extracted Spans :", extracted_spans)
print("Verbatim Checks :", verbatim_status)

print("\n=== STAGE 5: DEBERTA-V3 NLI VERIFICATION ===")
if premise:
    verification = verify_entailment(premise, target_claim)
    print("Premise (Localized Spans) :", premise)
    print("Hypothesis (Claim)        :", target_claim)
    print("Predicted NLI Label       :", verification["predicted_label"].upper())
    print("Entailment Confidence     :", f"{verification['entailment_prob'] * 100:.2f}%")
    print("Class Probabilities       :", verification["probs"])
else:
    print("No supporting spans localized from retrieved text.")

print("\n=== ERROR ANALYSIS AUDIT ===")
# Extract string titles from gold dictionaries or strings
gold_titles = [g["title"] if isinstance(g, dict) else g for g in gold_wikipages]
# Check for title overlap between retrieved articles and ASQA gold pages
title_overlap = any(
    any(gold.lower() in ret.lower() or ret.lower() in gold.lower() for gold in gold_titles)
    for ret in retrieved_titles
)
if not title_overlap:
    error_mode = "Retrieval Miss (Search API returned entity bio pages rather than comprehensive cross-player record lists)"
elif premise and verification.get("is_entailed"):
    error_mode = "Fully Supported (Claim substantiated by retrieved Wikipedia text)"
elif premise and not verification.get("is_entailed"):
    error_mode = "Temporal Drift / Scope Divergence (Live Wikipedia stats diverge from model training cutoff)"
else:
    error_mode = "Extraction Failure (Localizer could not isolate candidate evidence)"
print("Retrieved Titles :", retrieved_titles)
print("Gold Titles      :", gold_titles)
print("Gold Match Found :", title_overlap)
print("Audit Verdict    :", error_mode)

=== STAGE 1: CLOSED-BOOK GENERATION ===
Question: Who has the highest goals in world football?
Generated Answer:
 As of October 2023, the player with the highest number of official goals in world football is Cristiano Ronaldo. He has surpassed the 800-goal mark in official matches throughout his career, playing for clubs such as Manchester United, Real Madrid, and Juventus, as well as the Portugal national team. His goal tally continues to grow as he remains active in professional football.

=== STAGE 2: ATOMIC CLAIM DECOMPOSITION ===
Decomposed into 7 atomic claims:
  Claim 1: As of October 2023, Cristiano Ronaldo has the highest number of official goals in world football.
  Claim 2: Cristiano Ronaldo has surpassed the 800-goal mark in official matches throughout his career.
  Claim 3: Cristiano Ronaldo has played for Manchester United.
  Claim 4: Cristiano Ronaldo has played for Real Madrid.
  Claim 5: Cristiano Ronaldo has played for Juventus.
  Claim 6: Cristiano Ronaldo has played